# 04 — Export & Push Weights to GitHub
Copies the trained `.pth` weight files from Google Drive into the cloned repo's
`algo_trader/weights/` directory, then commits and pushes them using `gitpython`.
A timestamp tag (`weights-YYYY-MM-DD-HHmm`) is created so each training run is versioned.

**Prerequisites:**
- `GITHUB_TOKEN` must be added to Colab Secrets (🔑 icon) with `repo` scope.
- `GITHUB_USERNAME` and `GITHUB_REPO` must be set in the config cell below.

> ⚠️  This notebook pushes directly to the `main` branch.  
> For production use, push to a `weights` branch and open a PR.

In [ ]:
!pip install -q gitpython

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

# ─── EDIT THESE ──────────────────────────────────────────────────────────────
GITHUB_USERNAME = 'YOUR_GITHUB_USERNAME'
GITHUB_REPO     = 'deepscalper_copilot'
# ─────────────────────────────────────────────────────────────────────────────

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise RuntimeError('Add GITHUB_TOKEN (with repo scope) to Colab Secrets.')

DRIVE_WEIGHTS = '/content/drive/MyDrive/algo_trader/weights'
REPO_DIR      = '/content/deepscalper_copilot'
REPO_WEIGHTS  = f'{REPO_DIR}/algo_trader/weights'

import os
os.makedirs(REPO_WEIGHTS, exist_ok=True)
print('Paths configured ✓')

In [ ]:
import git

REMOTE_URL = (
    f'https://{GITHUB_USERNAME}:{GITHUB_TOKEN}'
    f'@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
)

if os.path.exists(REPO_DIR + '/.git'):
    repo = git.Repo(REPO_DIR)
    print('Repo already cloned — pulling latest changes...')
    origin = repo.remote('origin')
    origin.set_url(REMOTE_URL)
    origin.pull('main')
else:
    print('Cloning repo...')
    repo = git.Repo.clone_from(REMOTE_URL, REPO_DIR)

print(f'HEAD: {repo.head.commit.hexsha[:8]}  ({repo.head.commit.message.strip()})')

In [ ]:
import shutil
from pathlib import Path

pth_files = sorted(Path(DRIVE_WEIGHTS).glob('*.pth'))
print(f'Found {len(pth_files)} weight files in Drive:')

copied = []
for src in pth_files:
    dst = Path(REPO_WEIGHTS) / src.name
    shutil.copy2(src, dst)
    copied.append(str(dst))

print(f'Copied {len(copied)} files → {REPO_WEIGHTS}')

In [ ]:
from datetime import datetime

# Stage all .pth files
repo.index.add([str(Path(f).relative_to(REPO_DIR)) for f in copied])

if repo.is_dirty():
    timestamp   = datetime.utcnow().strftime('%Y-%m-%d-%H%M')
    tag_name    = f'weights-{timestamp}'
    commit_msg  = f'chore: push trained weights [{timestamp}] ({len(copied)} files)'

    # Configure git identity for the commit
    with repo.config_writer() as cfg:
        cfg.set_value('user', 'name',  GITHUB_USERNAME)
        cfg.set_value('user', 'email', f'{GITHUB_USERNAME}@users.noreply.github.com')

    commit = repo.index.commit(commit_msg)
    print(f'Committed: {commit.hexsha[:8]} — {commit_msg}')

    # Tag the commit
    repo.create_tag(tag_name, ref=commit)
    print(f'Tag created: {tag_name}')

    # Push commit and tag to GitHub
    origin = repo.remote('origin')
    origin.push('main')
    origin.push(tag_name)
    print(f'✅ Pushed to GitHub: {GITHUB_USERNAME}/{GITHUB_REPO} (tag: {tag_name})')
else:
    print('Nothing new to commit — repo is clean.')